# [9665] Homework 2 : Latent Dirichlet Allocation
Data files:
* https://raw.githubusercontent.com/vjavaly/Baruch-CIS-9665/main/data/Aus_legal_cases_500.csv
* https://raw.githubusercontent.com/vjavaly/Baruch-CIS-9665/main/data/Aus_legal_cases_15k.csv

## Homework Submission Rules (for all homework assignments)
* Homework is due by 6:05 PM on the due date
  * No late submission will be accepted
* Verify that you are submitting the correct homework file
* Homework file naming convention
  * LastName_FirstName_HwX.ipynb  [Replace X with the homework #]
    * 0.5 points deducted for submitting homework not complying with naming convention
* Before submission, execute "Kernel -> Restart Kernel and Run All Cells"
  * 0.5 points deducted for not submitting a cleanly executed notebook

## Homework 2 Requirements
* Load data [Aus_legal_cases_15k.csv] into dataframe
* Save indexes for at least 1 sample document
* Perform text preprocessing
* Generate Gensim dictionary
* Remove very rare and very common words
* Vectorize data
* Train LDA model on vectorized data
* Evaluate LDA model on at least 1 sample document

## Professor's suggestions
* Use the smaller data file [Aus_legal_cases_500.csv] to develop this notebook
* Once you have a working notebook, switch to the larger data file [Aus_legal_cases_15k.csv]
* Remember to submit this homework using the larger data file [Aus_legal_cases_15k.csv]

In [ ]:
from datetime import datetime
print(f'Run time: {datetime.now().strftime("%D %T")}')

Run time: 03/28/25 00:18:46


### Import libraries

In [ ]:
!pip install gensim

In [ ]:
import pandas as pd
import nltk
from gensim.utils import simple_preprocess
from gensim.parsing.preprocessing import STOPWORDS
from nltk.stem import WordNetLemmatizer
from gensim.corpora import Dictionary
from gensim.models import TfidfModel
from gensim.models import LdaModel
from gensim.models import LdaMulticore     # faster implementation of LDA (parallelized for multicore machines)
from gensim.models.coherencemodel import CoherenceModel
import multiprocessing

In [ ]:
!pip install --upgrade numpy gensim

  Using cached numpy-2.2.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (62 kB)


In [ ]:
nltk.download('wordnet')

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

### Load data

In [ ]:
pd.set_option('max_colwidth', None)

In [ ]:
# Load data file Aus_legal_cases_15k.csv into dataframe
cases_500 = pd.read_csv('https://raw.githubusercontent.com/vjavaly/Baruch-CIS-9665/main/data/Aus_legal_cases_500.csv')

### Examine data

In [ ]:
# displaying the shape
cases_500.shape

(500, 4)

In [ ]:
# displaying the head
cases_500.head(3)

case_id case_outcome  \
0  Case15689        cited   
1   Case9347      applied   
2   Case8483        cited   

                                                                    case_title  \
0                          Knight v Beyond Properties Pty Ltd [2007] FCAFC 170   
1                                        Powell v Evreniades (1989) 21 FCR 252   
2  Wigan v English and Scottish Law Life Assurance Association (1909) 1 Ch 291   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

### Save index for at least 1 sample document

In [ ]:
# Get index value for at least 1 sample case
sample_case_index = 13

### Create preprocessing function(s)

In [ ]:
# creating preprocessing functions
def lemmatizer(text):
    return WordNetLemmatizer().lemmatize(text, pos='v')

def preprocess(text):
    result = []
    for token in simple_preprocess(text):
        if token not in STOPWORDS and len(token) > 3:
            result.append(lemmatizer(token))
    return result

### Preprocess data

In [ ]:
# checking for null values
cases_500.isnull().sum()

,0
case_id,0
case_outcome,0
case_title,0
case_text,2


In [ ]:
# dropiing null values
cases_500 = cases_500.dropna()

In [ ]:
# Preprocess all documents
processed_docs = cases_500['case_text'].map(preprocess)

In [ ]:
# Review first few rows of preprocessed data
processed_docs[:3]

,case_text
0,"[relevant, principles, context, claim, breach, explain, court, australia, include, follow, initial, confusion, sufficient, necessary, confusion, induce, likely, induce, transaction, conduct, mislead, customer, mistake, impression, trader, connection, affiliation, open, negotiations, invite, approach, mislead, deceptive, true, position, emerge, transaction, conclude, case, depend, facts, conduct, mislead, deceptive, likely, mislead, deceive, notwithstanding, engender, temporary, commercially, irrelevant, error, knight, properties, fcafc, hansen, beverage, company, bickfords, australia, fcafc, conduct, mislead, deceptive, likely, mislead, deceive, involve, judgment, notional, cause, effect, relationship, conduct, putative, consumer, state, mind, addition, principles, clear, order, determine, contravention, necessary, determine, conduct, complain, amount, representation, likely, lead, misconception, arise, mind, section, public, conduct, direct, ...]"
1,"[term, hardship, define, powell, evreniades, evreniades, hill, consider, term, income, assessment, honour, concern, hardship, taxpayer, dependants, death, taxpayer, observe, definition, mean, hardship, expect, word, phrase, ordinary, english, word, have, understand, mean, context, word, appear, make, clear, relief, board, consider, exaction, involve, dependants, decease, taxpayer, financial, difficulty, circumstances, say, financial, difficulty, dependants, significant, need, warrant, action, relief, board, relieve, condition, hill, accept, distinction, extreme, hardship, note, constitute, depend, circumstances, honour, say, inappropriate, attempt, state, test, constitute, hardship, abstract, comment, clearly, severe, financial, hardship, dependants, decease, person, leave, destitute, mean, support, particular, case, constitute, hardship, observations, refer, apparent, approval, gummow, grieken, veilands, ...]"
2,"[antecedent, debt, pursuant, agreement, express, imply, constitute, valuable, consideration, include, bankruptcy, context, wigan, english, scottish, life, assurance, association, zampatti, parte, levack, cannane, anor, official, trustee, bankruptcy, official, trustee, bankruptcy, racovitis, victorian, producers, operative, kenneth, phillips, firm, westgold, resources, wasca, bankruptcy, relate, proceedings, forbearance, value, point, emphasise, para, explanatory, memorandum, bankruptcy, legislation, amendment, enact, current, form, state, forbearance, regard, good, consideration, forbearance, propose, amend, look, light, likely, value, choose, action, emphasis, add]"


### Generate Gensim Dictionary object

In [ ]:
%%time

# Map each word in ‘processed_docs’ to its unique integer id (index)
dictionary = Dictionary(processed_docs)

CPU times: user 70.3 ms, sys: 3.75 ms, total: 74 ms
Wall time: 74.2 ms


In [ ]:
# Remove very rare and very common words
#  Filter out tokens that appear in
#   < 15 documents (absolute number) or
#   > 50% documents (fraction of total corpus size, not absolute number)
#  After the above two steps, keep only the first 100000 most frequent tokens
dictionary.filter_extremes(no_below=15, no_above=0.5, keep_n=100000)
print("{} words remaining in Dictionary object".format(len(dictionary)))

724 words remaining in Dictionary object


In [ ]:
# Convert documents into the bag-of-words format: list of (token_id, token_count) 2-tuples
bow_corpus = [dictionary.doc2bow(doc) for doc in processed_docs]

In [ ]:
# checking preprocessed BoW version of sample document
print(bow_corpus[sample_case_index])

[(2, 6), (4, 1), (7, 2), (9, 1), (11, 1), (13, 1), (20, 1), (32, 4), (33, 1), (37, 3), (47, 1), (68, 2), (70, 1), (73, 1), (75, 2), (81, 1), (86, 6), (89, 1), (91, 1), (92, 1), (102, 1), (106, 1), (109, 1), (117, 1), (118, 1), (120, 2), (125, 1), (140, 1), (146, 3), (151, 4), (154, 1), (162, 1), (171, 2), (173, 1), (175, 2), (183, 1), (191, 1), (192, 1), (201, 4), (203, 16), (226, 1), (245, 1), (257, 1), (263, 1), (278, 17), (285, 1), (295, 1), (300, 4), (302, 1), (307, 4), (323, 1), (324, 1), (330, 1), (332, 8), (333, 3), (334, 8), (335, 6), (342, 1), (343, 1), (345, 7), (347, 1), (353, 3), (354, 1), (357, 1), (362, 1), (383, 3), (406, 1), (409, 1), (422, 1), (425, 2), (431, 1), (448, 2), (450, 1), (451, 2), (452, 1), (453, 1), (454, 2), (455, 1), (456, 1), (457, 1), (458, 1), (459, 1), (460, 1), (461, 3), (462, 1), (463, 1), (464, 1), (465, 3)]


In [ ]:
 # reviewing sample document in BoW format
 bow_example_doc = bow_corpus[sample_case_index]

print('word_index\tword\t\tword_occurences')
for i in range(len(bow_example_doc)):
    print(f'{bow_example_doc[i][0]}\t\t{dictionary[bow_example_doc[i][0]]}\t\t{bow_example_doc[i][1]}')

word_index	word		word_occurences
2		applicant		6
4		arise		1
7		australia		2
9		breach		1
11		claim		1
13		clear		1
20		context		1
32		fcafc		4
33		follow		1
37		include		3
47		limit		1
68		relation		2
70		relevant		1
73		respondent		1
75		section		2
81		account		1
86		affairs		6
89		answer		1
91		appeal		1
92		appear		1
102		circumstances		1
106		concern		1
109		consider		1
117		decision		1
118		define		1
120		determination		2
125		example		1
140		honour		1
146		leave		3
151		mean		4
154		merely		1
162		person		1
171		provide		2
173		purpose		1
175		receive		2
183		say		1
191		submissions		1
192		submit		1
201		think		4
203		tribunal		16
226		relate		1
245		basis		1
257		deal		1
263		doubt		1
278		information		17
285		material		1
295		possible		1
300		process		4
302		protection		1
307		rely		4
323		summarise		1
324		take		1
330		dispute		1
332		immigration		8
333		indigenous		3
334		minister		8
335		multicultural		6
342		absence		1
343		accordingly		1
345		appellant		7
347		conclusion

### Run TF-IDF on Bag of Words

In [ ]:
# Create tf-idf model object on ‘bow_corpus’
tfidf = TfidfModel(bow_corpus)

In [ ]:
# Apply transformation to entire corpus
tfidf_corpus = tfidf[bow_corpus]

In [ ]:
# reviewing sample document in TF-IDF format
print('word_index\tword\t\ttfidf_value')
for i in range(len(bow_example_doc)):
    print(f'{bow_example_doc[i][0]}\t\t{dictionary[bow_example_doc[i][0]]}\t\t{tfidf_corpus[sample_case_index][i][1]}')

word_index	word		tfidf_value
2		applicant		0.13045209514713788
4		arise		0.030047563963974094
7		australia		0.03635062945074209
9		breach		0.03765451271337978
11		claim		0.02089147354742401
13		clear		0.03204823585474688
20		context		0.030047563963974094
32		fcafc		0.10466346546614062
33		follow		0.02237829655578919
37		include		0.07849759909960548
47		limit		0.02460193755489353
68		relation		0.05543278634872148
70		relevant		0.02186727567797716
73		respondent		0.02753654989942907
75		section		0.06409647170949376
81		account		0.036059564802422873
86		affairs		0.15125639123634982
89		answer		0.04521959978599506
91		appeal		0.02633095299990775
92		appear		0.03158165809487144
102		circumstances		0.020540278401022984
106		concern		0.030047563963974094
109		consider		0.02019658265945715
117		decision		0.0211299342889564
118		define		0.04576527257310711
120		determination		0.08368795775566383
125		example		0.039425065744422304
140		honour		0.02536491477966502
146		leave		0.09832345683602077


### Train LDA model using Bag of Words corpus

In [ ]:
# Train model with Bag of Words corpus
num_cores = multiprocessing.cpu_count()
print(f'Number of cores: {num_cores}')

Number of cores: 2


In [ ]:
%%time

# training model with BoW corpus
lda_model = LdaMulticore(bow_corpus, num_topics=10, id2word=dictionary, passes=10, iterations=100, workers=num_cores)

# hyperparameters for later review
# corpus=bow_corpus, id2word=dictionary, num_topics=10,chunksize=10, passes=10, gamma_threshold=0.00, per_word_topics=True, workers=num_cores, random_state=42)


CPU times: user 10 s, sys: 501 ms, total: 10.5 s
Wall time: 11.8 s


In [ ]:
# For each topic, explore the words occuring in that topic and its relative weight
for idx, topic in lda_model.print_topics():
    print('Topic # {}: {}\n'.format(idx, topic))

Topic # 0: 0.035*"title" + 0.035*"tribunal" + 0.026*"land" + 0.024*"evidence" + 0.020*"right" + 0.016*"applicant" + 0.014*"interest" + 0.014*"group" + 0.013*"people" + 0.012*"exist"

Topic # 1: 0.057*"tribunal" + 0.054*"minister" + 0.044*"immigration" + 0.037*"affairs" + 0.033*"multicultural" + 0.021*"indigenous" + 0.019*"appellant" + 0.018*"decision" + 0.018*"applicant" + 0.016*"evidence"

Topic # 2: 0.031*"order" + 0.021*"cost" + 0.019*"party" + 0.015*"conduct" + 0.011*"respondents" + 0.009*"australian" + 0.009*"say" + 0.008*"issue" + 0.008*"person" + 0.008*"circumstances"

Topic # 3: 0.020*"mean" + 0.016*"claim" + 0.013*"goods" + 0.013*"word" + 0.013*"commissioner" + 0.012*"australia" + 0.012*"construction" + 0.012*"purpose" + 0.008*"scheme" + 0.008*"say"

Topic # 4: 0.036*"document" + 0.029*"privilege" + 0.022*"legal" + 0.014*"evidence" + 0.014*"commissioner" + 0.014*"claim" + 0.010*"issue" + 0.010*"applicant" + 0.010*"respondent" + 0.009*"order"

Topic # 5: 0.021*"contract" + 0.01

#### Evaluate LDA model trained on Bag of Words corpus

In [ ]:
%%time

# Compute Coherence Score: c_v
coherence_model_lda = CoherenceModel(model=lda_model, texts=processed_docs,
                                     dictionary=dictionary, coherence='c_v')
coherence_lda = coherence_model_lda.get_coherence()
print('LDA BOW Coherence Score: ', coherence_lda)

LDA BOW Coherence Score:  0.32498590064005095
CPU times: user 2.26 s, sys: 34.5 ms, total: 2.3 s
Wall time: 2.31 s


#### Check LDA BoW model topics on sample document

In [ ]:
# chekcing sample documnent
print(processed_docs[sample_case_index])

['support', 'gain', 'know', 'ticket', 'case', 'contract', 'case', 'establish', 'exemption', 'clause', 'contain', 'ticket', 'party', 'actually', 'aware', 'term', 'clause', 'contract', 'party', 'seek', 'rely', 'time', 'contract', 'reasonable', 'notice', 'give', 'party', 'seddon', 'ellinghaus', 'cheshire', 'fifoot', 'contract', 'australian', 'edition', 'oceanic', 'line', 'special', 'ship', 'company', 'principle', 'application', 'author', 'cheshire', 'fifoot', 'observe', 'necessary', 'determine', 'precision', 'point', 'contract', 'present', 'case', 'composers', 'need', 'establish', 'sinclair', 'submit', 'entry', 'publication', 'circular', 'july', 'edition', 'matilda', 'publications', 'constitute', 'reasonable', 'notice', 'have', 'regard', 'sinclair', 'close', 'connection', 'victorian', 'girl', 'guide', 'mere', 'conjecture', 'happen', 'equally', 'probable', 'hear', 'competition', 'word', 'mouth', 'submit', 'entry', 'publication', 'document', 'accordingly', 'establish', 'evidence', 'sinclair

In [ ]:
# converting sample document into BoW format: list of (token_id, token_count) 2-tuples
sample_bow = dictionary.doc2bow(processed_docs[sample_case_index])

topics = [[word for word, prob in lda_model.show_topic(topicid, topn=5)] \
          for topicid in range(lda_model.num_topics)]

In [ ]:
# Get topic distribution for the sample document
topic_distribution = lda_model.get_document_topics(sample_bow)

In [ ]:
# Display topic distribution for sample document
topic_distribution

[(0, 0.15066926), (2, 0.051555805), (5, 0.70184773), (7, 0.08683365)]

### Train LDA model using TF-IDF corpus

In [ ]:
%%time

# Train model with TFIDF corpus
lda_model_tfidf = LdaMulticore(corpus=tfidf_corpus, id2word=dictionary, num_topics=10,
                               chunksize=10, passes=10, gamma_threshold=0.001,
                               per_word_topics=True, workers=num_cores, random_state=42)

CPU times: user 2.99 s, sys: 211 ms, total: 3.2 s
Wall time: 4.84 s


In [ ]:
# For each topic, explore the words occuring in that topic and its relative weight
for idx, topic in lda_model_tfidf.print_topics():
    print('Topic # {}: {}\n'.format(idx, topic))


Topic # 0: 0.013*"applicant" + 0.012*"order" + 0.011*"claim" + 0.011*"evidence" + 0.010*"commissioner" + 0.009*"trade" + 0.009*"say" + 0.009*"rule" + 0.009*"question" + 0.008*"australia"

Topic # 1: 0.038*"house" + 0.035*"insurance" + 0.033*"broadcast" + 0.032*"gibbs" + 0.028*"city" + 0.028*"investments" + 0.019*"satisfaction" + 0.019*"legislative" + 0.019*"additional" + 0.018*"life"

Topic # 2: 0.050*"lord" + 0.037*"bankruptcy" + 0.034*"contract" + 0.033*"smith" + 0.025*"land" + 0.025*"judge" + 0.025*"society" + 0.022*"security" + 0.021*"council" + 0.019*"observation"

Topic # 3: 0.047*"interlocutory" + 0.036*"maker" + 0.029*"find" + 0.025*"explanation" + 0.023*"objective" + 0.022*"administrative" + 0.022*"days" + 0.018*"essential" + 0.018*"reach" + 0.017*"facie"

Topic # 4: 0.068*"tribunal" + 0.059*"minister" + 0.058*"immigration" + 0.054*"multicultural" + 0.051*"affairs" + 0.040*"indigenous" + 0.032*"appellant" + 0.022*"ground" + 0.020*"raise" + 0.020*"grind"

Topic # 5: 0.038*"good

In [ ]:
# using show_topics, which gives more options
topics = lda_model_tfidf.show_topics(num_topics=5, num_words=5, formatted=False)
for topic in topics:
    print(topic)

(5, [('goods', 0.03806585), ('association', 0.03502881), ('business', 0.027737124), ('international', 0.024630366), ('employee', 0.02227289)])
(9, [('distinguish', 0.043289058), ('damage', 0.027852615), ('prospect', 0.025479417), ('copyright', 0.023457699), ('infringement', 0.0229634)])
(1, [('house', 0.03775161), ('insurance', 0.034969952), ('broadcast', 0.033047073), ('gibbs', 0.03224215), ('city', 0.028324435)])
(7, [('respondents', 0.023502573), ('application', 0.022199327), ('applicants', 0.02185319), ('plaintiff', 0.021585012), ('person', 0.021432772)])
(0, [('applicant', 0.012541305), ('order', 0.011763197), ('claim', 0.011247145), ('evidence', 0.010759902), ('commissioner', 0.009777702)])



#### Evaluate LDA model trained on TF-IDF corpus

In [ ]:
%%time

# Compute Coherence Score: c_v
coherence_model_lda = CoherenceModel(model=lda_model_tfidf, texts=processed_docs,
                                     dictionary=dictionary, coherence='c_v')
coherence_lda = coherence_model_lda.get_coherence()
print('LDA TFIDF Coherence Score: ', coherence_lda)

LDA TFIDF Coherence Score:  0.3177123522344419
CPU times: user 2.68 s, sys: 30.6 ms, total: 2.71 s
Wall time: 2.79 s


#### Check LDA TF-IDF model topics on sample document

In [ ]:
topics = [[word for word, prob in lda_model_tfidf.show_topic(topicid, topn=5)] \
          for topicid in range(lda_model_tfidf.num_topics)]

# Get topic distribution for the sample document
topic_distribution = lda_model_tfidf.get_document_topics(sample_bow)

In [ ]:
# Display topic distribution for sample document
topic_distribution

[(0, 0.52581215),
 (2, 0.15037529),
 (6, 0.1761857),
 (7, 0.10819954),
 (8, 0.016709786),
 (9, 0.016656399)]